In [19]:
import arcpy
import pandas as pd

In [20]:
# Define paths
gdb_path = r"C:\National Water Utility\test.gdb"
excel_path = r"C:\National Water Utility\nwsc_gdb_domains.xlsx"
sheet_name = "Domains"  # Name of the tab in Excel

# Read the Excel sheet using pandas
df = pd.read_excel(excel_path, sheet_name=sheet_name)

# Get a list of existing domains in the GDB to avoid duplicate errors
existing_domains = [d.name for d in arcpy.da.ListDomains(gdb_path)]

# Group the data by Domain Name to process one domain at a time
grouped = df.groupby('DomainName')

for domain_name, group in grouped:
    # Extract domain metadata from the first row of the group
    first_row = group.iloc[0]
    domain_type = str(first_row['DomainType']).upper().strip()  # CODED or RANGE
    field_type = str(first_row['FieldType']).upper().strip()    # TEXT, SHORT, LONG, etc.
    
    print(f"Processing domain: {domain_name} ({domain_type})...")
    
    # 1. Create the Domain container if it doesn't exist
    if domain_name not in existing_domains:
        arcpy.management.CreateDomain(
            in_workspace=gdb_path,
            domain_name=domain_name,
            domain_description=f"Imported {domain_name} domain",
            field_type=field_type,
            domain_type=domain_type
        )
        print(f" -> Created empty domain: {domain_name}")
    else:
        print(f" -> Domain {domain_name} already exists. Adding/updating values.")

    # 2. Populate the Domain based on its type
    if domain_type == "CODED":
        for index, row in group.iterrows():
            code = row['CodeOrMin']
            description = row['DescriptionOrMax']
            
            # Add coded value (wrapped in try/except in case code already exists)
            try:
                arcpy.management.AddCodedValueToDomain(
                    in_workspace=gdb_path,
                    domain_name=domain_name,
                    code=code,
                    code_description=description
                )
            except arcpy.ExecuteError as e:
                # Catches error if the specific code already exists in that domain
                print(f"    ! Code '{code}' could not be added (may already exist).")
                
    elif domain_type == "RANGE":
        # For a range domain, we just need the min and max from the first row
        min_val = first_row['CodeOrMin']
        max_val = first_row['DescriptionOrMax']
        
        arcpy.management.SetValueForRangeDomain(
            in_workspace=gdb_path,
            domain_name=domain_name,
            min_value=min_val,
            max_value=max_val
        )
        print(f" -> Set range boundary to: {min_val} - {max_val}")

print("\nAll domains processed successfully!")

Processing domain: Bulk_meter_sizes (CODED)...
 -> Domain Bulk_meter_sizes already exists. Adding/updating values.
    ! Code 'NWSCU135' could not be added (may already exist).
    ! Code 'NWSCU136' could not be added (may already exist).
    ! Code 'NWSCU137' could not be added (may already exist).
    ! Code 'NWSCU138' could not be added (may already exist).
    ! Code 'NWSCU139' could not be added (may already exist).
    ! Code 'NWSCU140' could not be added (may already exist).
    ! Code 'NWSCU141' could not be added (may already exist).
    ! Code 'NWSCU142' could not be added (may already exist).
    ! Code 'NWSCU143' could not be added (may already exist).
    ! Code 'NWSCU144' could not be added (may already exist).
    ! Code 'NWSCU145' could not be added (may already exist).
    ! Code 'NWSCU146' could not be added (may already exist).
    ! Code 'NWSCU147' could not be added (may already exist).
    ! Code 'NWSCU148' could not be added (may already exist).
    ! Code 'NWSCU